# Output a program can read

MichAl Academy, unit 5.2.

Run each cell with **Shift+Enter**.

A model writes text. Software needs fields. Everything in this unit is about
the join between those two facts: how to get a reply your code can load
without guessing, what a function call is underneath, and which of the errors
a check can actually catch.

The model is **SmolLM2-135M-Instruct** again, small and therefore honest. Every
failure here happens at every size; it happens more often here, which is what
makes it visible in twenty-four items instead of twenty-four thousand.


In [ ]:
import json
import re
import time
import warnings

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME)
model.eval()


def chat(messages, prefill=""):
    """The prompt as one string, so we can append text the model must continue."""
    return tok.apply_chat_template(messages, add_generation_prompt=True,
                                   tokenize=False) + prefill


@torch.no_grad()
def generate(prompt, n=32):
    ids = tok(prompt, return_tensors="pt").input_ids
    out = model.generate(ids, max_new_tokens=n, do_sample=False,
                         pad_token_id=tok.eos_token_id,
                         attention_mask=torch.ones_like(ids))
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)


print(generate(chat([{"role": "user", "content": "Say the word ready."}])))


## The task and the shape

The twenty-four support messages from unit 5.1, and this time the answer has to
arrive as an object with two fields: which of three labels, and whether it reads
as urgent.

Three things get scored, and keeping them apart is the whole unit:

- **parses** — `json.loads` accepts it
- **schema** — it has both keys, the label is one of the three, urgent is a
  boolean
- **correct** — the label is the right one

A reply can pass the first two and fail the third. That is the point.


In [ ]:
LABELS = ["billing", "delivery", "technical"]

ITEMS = [
    ("My card was charged twice for the same order.", "billing"),
    ("The parcel says delivered but nothing arrived.", "delivery"),
    ("The app crashes every time I open the settings.", "technical"),
    ("I was invoiced for a subscription I cancelled in March.", "billing"),
    ("Tracking has not updated in six days.", "delivery"),
    ("I cannot log in, it says my password is wrong after a reset.", "technical"),
    ("Why is there a 4 pound fee on my statement?", "billing"),
    ("The courier left the box at the wrong house number.", "delivery"),
    ("Video playback stops after ten seconds on wifi.", "technical"),
    ("Please refund the difference after the price drop.", "billing"),
    ("My order shipped to my old address.", "delivery"),
    ("The export button does nothing in Firefox.", "technical"),
    ("You took the annual fee again after I downgraded.", "billing"),
    ("The driver marked it undeliverable without knocking.", "delivery"),
    ("Two-factor codes never arrive on my phone.", "technical"),
    ("I was promised a credit note and it never appeared.", "billing"),
    ("It has been sitting at the sorting centre since Tuesday.", "delivery"),
    ("The search results page loads forever and then times out.", "technical"),
    ("The discount code was not applied at checkout.", "billing"),
    ("Half the order arrived and the rest is missing.", "delivery"),
    ("Uploading a photo fails with an unknown error.", "technical"),
    ("My statement shows a currency conversion charge nobody mentioned.", "billing"),
    ("The estimated date has moved back three times.", "delivery"),
    ("The mobile app logs me out every few minutes.", "technical"),
]

TASK = "Classify the support message as one of: billing, delivery, technical."


def grade(text, gold):
    """Returns (parses, schema, correct) for one reply."""
    found = re.search(r"\{.*?\}", text, re.S)
    if not found:
        return 0, 0, 0
    try:
        obj = json.loads(found.group(0))
    except ValueError:
        return 0, 0, 0
    schema = (isinstance(obj, dict)
              and obj.get("label") in LABELS
              and isinstance(obj.get("urgent"), bool))
    return 1, int(schema), int(obj.get("label") == gold)


def run(build, prefix="", n=32):
    totals = [0, 0, 0]
    first = []
    for text, gold in ITEMS:
        reply = prefix + generate(build(text), n=n)
        for i, v in enumerate(grade(reply, gold)):
            totals[i] += v
        if len(first) < 2:
            first.append(reply.strip()[:80])
    return totals, first


## Asking for the shape in words

Two prompts. One describes the shape as a rule, the way a schema is written.
The other shows one filled-in example of it.


In [ ]:
RULE = ('Reply with JSON only, in exactly this shape:\n'
        '{"label": "billing" or "delivery" or "technical", '
        '"urgent": true or false}')

EXAMPLE = ('Reply with JSON only, in exactly this shape:\n'
           '{"label": "delivery", "urgent": false}')


def described(text):
    return chat([{"role": "user", "content": f"{TASK}\n\n{RULE}\n\nMessage: {text}"}])


def shown(text):
    return chat([{"role": "user", "content": f"{TASK}\n\n{EXAMPLE}\n\nMessage: {text}"}])


print(f"{'':<22}{'parses':>8}{'schema':>8}{'correct':>9}")
for name, build in (("shape described", described), ("shape shown filled in", shown)):
    (p, s, c), first = run(build)
    print(f"{name:<22}{p:>6}/24{s:>6}/24{c:>7}/24")
    for reply in first:
        print(f"{'':<24}{reply!r}")


**The described shape came back verbatim.** Not filled in: copied, `or` and all.
The model was reading it as text to continue rather than as a rule to obey, and
unit 4.4.1 explains why — continuing text is the only thing it was trained to
do.

**One filled-in example fixed most of it.** Same information, written as a thing
rather than as a rule about things.

That is the first practical lesson of the unit and it is cheap: **show an
instance, not a grammar.**


## Making the first characters yours

The reply has to start with `{"label": "`. Nothing says the model has to be the
one that writes it. A chat prompt is just text, so we can end it mid-object and
let the model continue from there.

This is the same trick under every "JSON mode" you have used, and the same one
5.1 measured for examples: **put the shape where the model is continuing it, not
where it is reading about it.**


In [ ]:
def prefilled(text):
    return chat([{"role": "user", "content": f"{TASK}\n\n{EXAMPLE}\n\nMessage: {text}"}],
                prefill='{"label": "')


SHOTS = [
    ("The payment failed but the money left my account.",
     '{"label": "billing", "urgent": true}'),
    ("It has been stuck at the depot for a week.",
     '{"label": "delivery", "urgent": false}'),
    ("The page is blank after I sign in.",
     '{"label": "technical", "urgent": false}'),
]


def as_turns(text):
    msgs = [{"role": "user", "content": f"{TASK}\n\n{EXAMPLE}\n\nMessage: {SHOTS[0][0]}"},
            {"role": "assistant", "content": SHOTS[0][1]}]
    for message, answer in SHOTS[1:]:
        msgs.append({"role": "user", "content": f"Message: {message}"})
        msgs.append({"role": "assistant", "content": answer})
    msgs.append({"role": "user", "content": f"Message: {text}"})
    return chat(msgs)


print(f"{'':<22}{'parses':>8}{'schema':>8}{'correct':>9}")
for name, build, prefix in (
    ("started for it", prefilled, '{"label": "'),
    ("3 examples as turns", as_turns, ""),
):
    (p, s, c), first = run(build, prefix)
    print(f"{name:<22}{p:>6}/24{s:>6}/24{c:>7}/24")


Before reading anything into those numbers, look at what was actually answered.


In [ ]:
from collections import Counter


def labels_from(build, prefix="", n=32):
    got = Counter()
    for text, _ in ITEMS:
        reply = prefix + generate(build(text), n=n)
        found = re.search(r"\{.*?\}", reply, re.S)
        try:
            got[json.loads(found.group(0))["label"]] += 1
        except (AttributeError, ValueError, KeyError):
            got["unreadable"] += 1
    return got


print("the messages themselves:", dict(Counter(g for _, g in ITEMS)))
for name, build, prefix in (("one example", shown, ""),
                            ("started for it", prefilled, '{"label": "'),
                            ("3 examples as turns", as_turns, "")):
    print(f"{name:<22}", dict(labels_from(build, prefix)))


**Every one of them answers with a single label.** Eight messages of each kind
were put in on purpose, so a model that always says "delivery" scores 8 of 24
without reading anything, and a model that always says "technical" scores the
same. That is where 7, 8 and 10 came from.

So the format was bought completely and the task was not attempted. This is the
sharpest version of 5.1.2's finding: examples teach the **shape** of an answer,
and this model took the shape and the content together.


Is that the examples' fault or the object's? Unit 5.1 ran the same three
examples, as the same conversation turns, on the same twenty-four messages, and
asked for a bare label instead of an object. Run that here and compare.


In [ ]:
def bare_label(text):
    """5.1's version: same examples as turns, answer is the label alone."""
    rule = ("Classify the support message as one of: billing, delivery, technical.\n"
            "Answer with the single label and nothing else.")
    pairs = [(m, json.loads(a)["label"]) for m, a in SHOTS]
    msgs = [{"role": "user", "content": f"{rule}\n\nMessage: {pairs[0][0]}"},
            {"role": "assistant", "content": pairs[0][1]}]
    for message, label in pairs[1:]:
        msgs.append({"role": "user", "content": f"Message: {message}"})
        msgs.append({"role": "assistant", "content": label})
    msgs.append({"role": "user", "content": f"Message: {text}"})
    return chat(msgs)


got = Counter()
right = 0
for text, gold in ITEMS:
    reply = generate(bare_label(text), n=8).strip().strip(".").strip().lower()
    got[reply if reply in LABELS else "not a label"] += 1
    right += reply == gold
print("bare label, same examples:", dict(got), f" correct {right}/24")


**Same examples, same messages, and the answers vary.** Asked for a label the
model uses more than one; asked for the object it settles on a single answer and
repeats it.

That is the cost of the format, and it is the same trade 5.1.1 measured in the
other direction: the format rule bought compliance and lost accuracy. Here it
buys perfect compliance and loses the task.


## Not asking at all

Everything so far asks. A decoder does not have to ask, because it chooses the
next token from a probability over the whole vocabulary and **we own that
choice**. Set the probability of every token that would break the shape to zero
and the model cannot produce a broken one.

Here the legal outputs are six strings: three labels crossed with two booleans.
At each step, allow only tokens that keep the text a prefix of one of them, then
take the best allowed token.

This is what a library like Outlines or llama.cpp's grammar does, in eleven
lines and with no library.


In [ ]:
VOCAB = [tok.decode([i]) for i in range(len(tok))]
TARGETS = [f'{{"label": "{lab}", "urgent": {flag}}}'
           for lab in LABELS for flag in ("true", "false")]
allowed_cache = {}


def allowed_after(done):
    """Token ids that keep the output a prefix of some legal string."""
    if done not in allowed_cache:
        live = [t for t in TARGETS if t.startswith(done)]
        allowed_cache[done] = torch.tensor(
            [i for i, piece in enumerate(VOCAB)
             if piece and any(t.startswith(done + piece) for t in live)])
    return allowed_cache[done]


@torch.no_grad()
def constrained(prompt):
    ids = tok(prompt, return_tensors="pt").input_ids
    done = ""
    while done not in TARGETS:
        allowed = allowed_after(done)
        if len(allowed) == 0:
            break
        logits = model(ids).logits[0, -1]
        best = int(allowed[logits[allowed].argmax()])
        done += VOCAB[best]
        ids = torch.cat([ids, torch.tensor([[best]])], dim=1)
    return done


started = time.time()
totals = [0, 0, 0]
for text, gold in ITEMS:
    reply = constrained(chat([{"role": "user",
                               "content": f"{TASK}\n\n{EXAMPLE}\n\nMessage: {text}"}]))
    for i, v in enumerate(grade(reply, gold)):
        totals[i] += v
p, s, c = totals
print(f"{'constrained decode':<22}{p:>6}/24{s:>6}/24{c:>7}/24"
      f"   {time.time() - started:.0f}s")


How tight is the constraint? Count the legal tokens at each step of one reply.


In [ ]:
path = '{"label": "delivery", "urgent": false}'
done = ""
print(f"vocabulary: {len(VOCAB)} tokens\n")
while done != path:
    legal = allowed_after(done)
    taken = max((VOCAB[int(i)] for i in legal if path.startswith(done + VOCAB[int(i)])),
                key=len)
    print(f"{len(legal):>6} legal   after {done!r:<42} took {taken!r}")
    done += taken


**The whole object is twelve steps, and never more than thirteen of the 49,152
tokens are allowed to be next.** At the last step exactly one is: the closing
brace, because every legal string ends there.


**Twenty-four out of twenty-four parse, and that number is not a result.** It is
arithmetic: no sequence of allowed tokens spells anything else, so the only way
to score less than 24 is a bug in the decoder.

**The correct column did not move.** Compare it against the asking rows above.
Constraining the decoder bought the shape and did not buy a single right answer,
because nothing about masking the vocabulary tells the model what the message
was about.

Keep those two facts next to each other. A guaranteed schema is worth having and
it is worth exactly one thing.


## Calling a function

Same mechanism, different use. Unit 5.1 measured this model on arithmetic and
got 1 of 12 in every condition it tried, which is a floor rather than a result:
the model cannot do the sum.

So do not ask it to. Ask it for the **arguments**, and let Python do the sum.


In [ ]:
QUESTIONS = [
    ("I ordered 3 shirts at 14 pounds each and paid 5 pounds postage.", 3, 14, 5),
    ("Two chairs at 45 pounds each, delivery was 12 pounds.", 2, 45, 12),
    ("I bought 6 mugs at 7 pounds each with 4 pounds shipping.", 6, 7, 4),
    ("Four filters at 23 pounds each plus 9 pounds carriage.", 4, 23, 9),
    ("I took 5 cables at 8 pounds each and postage was 3 pounds.", 5, 8, 3),
    ("Ten pens at 2 pounds each, 6 pounds delivery.", 10, 2, 6),
    ("Three lamps at 31 pounds each with 15 pounds shipping.", 3, 31, 15),
    ("Seven books at 11 pounds each plus 8 pounds postage.", 7, 11, 8),
    ("Two monitors at 120 pounds each, delivery 20 pounds.", 2, 120, 20),
    ("Nine sockets at 6 pounds each and 7 pounds carriage.", 9, 6, 7),
    ("Five tiles at 19 pounds each with 25 pounds delivery.", 5, 19, 25),
    ("Eight brushes at 4 pounds each plus 6 pounds postage.", 8, 4, 6),
]


def total(quantity, unit_price, postage):
    """The tool. Ordinary Python, and it is never wrong."""
    return quantity * unit_price + postage


right = 0
for text, q, price, post in QUESTIONS:
    reply = generate(chat([{"role": "user", "content":
                            f"{text}\n\nWhat did the order cost in total? "
                            "Answer with the number only."}]), n=24)
    numbers = re.findall(r"\d+", reply.replace(",", ""))
    right += bool(numbers) and int(numbers[0]) == total(q, price, post)
print(f"answered straight out    {right:>2}/12")


A function call is not a new ability. It is the model writing an object that
names a function and its arguments, your code noticing it, running the function,
and the answer coming from Python. The only part the model does is fill in the
fields, which is the thing this unit already knows how to get.


In [ ]:
TOOL = ("You can call one function:\n"
        "total(quantity, unit_price, postage) -> the order total")

CALL_SHOTS = [
    ("I ordered 7 tables at 60 pounds each and paid 25 pounds postage.",
     '{"quantity": 7, "unit_price": 60, "postage": 25}'),
    ("Three hats at 9 pounds each, delivery was 2 pounds.",
     '{"quantity": 3, "unit_price": 9, "postage": 2}'),
    ("I bought 12 bolts at 1 pound each with 40 pounds shipping.",
     '{"quantity": 12, "unit_price": 1, "postage": 40}'),
]


def call_prompt(text):
    msgs = [{"role": "user", "content": f"{TOOL}\n\nMessage: {CALL_SHOTS[0][0]}"},
            {"role": "assistant", "content": CALL_SHOTS[0][1]}]
    for message, answer in CALL_SHOTS[1:]:
        msgs.append({"role": "user", "content": f"Message: {message}"})
        msgs.append({"role": "assistant", "content": answer})
    msgs.append({"role": "user", "content": f"Message: {text}"})
    return chat(msgs, prefill='{"quantity": ')


FIELDS = ("quantity", "unit_price", "postage")
calls = []
for text, q, price, post in QUESTIONS:
    raw = '{"quantity": ' + generate(call_prompt(text), n=28)
    found = re.search(r"\{.*?\}", raw, re.S)
    call = None
    if found:
        try:
            obj = json.loads(found.group(0))
            if all(isinstance(obj.get(k), int) for k in FIELDS):
                call = {k: obj[k] for k in FIELDS}
        except ValueError:
            pass
    calls.append((call, (q, price, post), raw))

usable = [c for c, _, _ in calls if c]
arguments = sum(c is not None and tuple(c.values()) == want
                for c, want, _ in calls)
answers = sum(c is not None and total(**c) == total(*want)
              for c, want, _ in calls)
print(f"produced a usable call   {len(usable):>2}/12")
print(f"arguments all correct    {arguments:>2}/12")
print(f"final answer correct     {answers:>2}/12")


**The arithmetic stopped being the problem and the reading started being it.**
Zero right on its own; with a function to call, the answer is right exactly as
often as the arguments are, because `total` cannot make a mistake.

Look at what the wrong calls got wrong.


In [ ]:
for call, want, raw in calls:
    if call is None:
        print(f"unusable   {raw.strip()[:64]!r}")
    elif tuple(call.values()) != want:
        print(f"wrong      {tuple(call.values())}  should be {want}")


Mostly it takes a number from the wrong place: the quantity repeated as the
price, or the postage read as the price. Every one of those is a **valid call**.
It names a real function, the argument names are right, the types are right, and
running it returns a number that looks exactly like an answer.


## What a check can catch

Three layers, and they do not catch the same things.

1. **Parse.** Is it JSON at all.
2. **Schema.** Are the keys, types and allowed values right.
3. **Something outside the model.** Anything about whether the values are *true*.

The first two are free and mechanical. Count what they leave.


In [ ]:
schema_ok = len(usable)
print(f"passed parse and schema  {schema_ok:>2}/12")
print(f"of those, actually right {arguments:>2}/{schema_ok}")
print(f"valid and wrong          {schema_ok - arguments:>2}/12")


A schema is a claim about shape. It has nothing to say about truth, and the
errors it lets through are the expensive ones, because they arrive looking
correct and get written to your database.

Layer three has to know something the model does not. Two candidates, measured
against the same eleven calls.


In [ ]:
def one_field(text, question):
    """Candidate one: ask the same model again, a different way."""
    reply = generate(chat([{"role": "user", "content":
                            f"{text}\n\n{question} Answer with the number only."}]),
                     n=8)
    numbers = re.findall(r"\d+", reply.replace(",", ""))
    return int(numbers[0]) if numbers else None


ASKS = (("quantity", "How many items were ordered?"),
        ("unit_price", "What does one item cost in pounds?"),
        ("postage", "What is the postage in pounds?"))

kept_right = kept_wrong = dropped_right = 0
for (call, want, _), (text, *_) in zip(calls, QUESTIONS):
    if call is None:
        continue
    second = {key: one_field(text, question) for key, question in ASKS}
    agrees = all(second[key] == call[key] for key in FIELDS)
    right = tuple(call.values()) == want
    kept_right += agrees and right
    kept_wrong += agrees and not right
    dropped_right += right and not agrees
print(f"calls it keeps, and they are right  {kept_right:>2}")
print(f"calls it keeps that are wrong       {kept_wrong:>2}")
print(f"right calls it throws away          {dropped_right:>2}/{arguments}")


It kept one call and that call was wrong, and it threw away all three of the
right ones. The second asking is not an independent opinion: it is the same
model, with the same weaknesses, reading the same sentence, so where it goes
wrong it tends to go wrong the same way twice, and where it goes right it does
not reliably do that twice either.

The other candidate knows the sentence shape and does not involve the model at
all.


In [ ]:
WORDS = {"two": 2, "three": 3, "four": 4, "five": 5, "six": 6,
         "seven": 7, "eight": 8, "nine": 9, "ten": 10}
CARRIER = "postage|shipping|delivery|carriage"


def by_rule(text):
    """Candidate two: read the three numbers with regular expressions."""
    price = re.search(r"(\d+) pounds? each", text)
    first = re.search(r"\b(\d+|" + "|".join(WORDS) + r")\b", text, re.I)
    post = (re.search(r"(\d+) pounds? (?:" + CARRIER + ")", text)
            or re.search(r"(?:" + CARRIER + r")(?: was)? (\d+) pounds?", text))
    if not (price and first and post):
        return None
    word = first.group(1).lower()
    return {"quantity": WORDS.get(word) or int(word),
            "unit_price": int(price.group(1)),
            "postage": int(post.group(1))}


caught = waved_through = false_alarms = 0
for (call, want, _), (text, *_) in zip(calls, QUESTIONS):
    if call is None:
        continue
    wrong = tuple(call.values()) != want
    disagrees = by_rule(text) != call
    caught += wrong and disagrees
    waved_through += wrong and not disagrees
    false_alarms += disagrees and not wrong
print(f"wrong calls the rule reader flags   {caught:>2}/{schema_ok - arguments}")
print(f"wrong calls it waves through        {waved_through:>2}")
print(f"correct calls it rejects            {false_alarms:>2}/{arguments}")


Every wrong call, flagged, and no correct one rejected. That is what layer three
looks like when it works: a source of truth the model had no hand in.

**And then read the bill.** Those regular expressions were written by looking at
these twelve sentences. They score 12 of 12 by themselves, better than the
model, and they are fitted to a shape rather than reading a sentence. Try a
thirteenth.


In [ ]:
THIRTEENTH = "The order was for a dozen hinges, priced at 3 pounds apiece, and they charged 8 for delivery."
print("rule reader on a new phrasing:", by_rule(THIRTEENTH))
print("should be:                    ", {"quantity": 12, "unit_price": 3, "postage": 8})


That is the trade in one line. **Where you can write the check, you may not need
the model. Where you cannot, the check has to come from somewhere else** — a
price list you own, a database the identifier has to exist in, a person
confirming before the money moves. What you must not do is what a schema
quietly invites: treat well-formed as true.


## What this unit measured

- **Describing the shape got the description back.** The model copied the
  schema, `or` and all. One filled-in example got the shape instead.
- **Starting the reply for the model** puts the format where it is being
  continued rather than read about, which is 5.1's placement finding in a new
  place.
- **Constraining the decoder makes the shape arithmetic rather than luck**, and
  it buys nothing else: the correct column sat where it already was.
- **A function removes the work the model cannot do** and leaves the work it
  can half do. The arithmetic went from the failure to not being a failure mode
  at all, and filling the arguments became the whole risk.
- **Most wrong calls are valid calls.** Parse and schema checks catch shape.
  Everything else needs a source of truth that is not the model.
